In [ ]:
from pathlib import Path
import pandas as pd
import os

In [ ]:
current_dir = Path.cwd()

# Input: Staging Data
staging_path = (current_dir.parent.parent / "data" / "staging" / "wipo" / "ip_indicators").resolve()

# Output: Intermediate Data (Target Folder)
intermediate_path = (current_dir.parent.parent / "data" / "intermediate").resolve()
intermediate_path.mkdir(parents=True, exist_ok=True)

print(f"Reading from: {staging_path}")
print(f"Exporting to: {intermediate_path}")

In [ ]:
all_dfs = []

# Get all parquet files
files = list(staging_path.glob("stg_wipo__*.parquet"))

print(f"\nScanning {len(files)} files in staging...")

for file_path in files:
    filename = file_path.stem  
    
    
    try:
        
        content_part = filename.split("__")[1] 
        
        
        if "_" not in content_part:
           
            continue
            
        parts = content_part.split("_")
        
        
        if len(parts) < 2:
            continue
            
        indicator_code = parts[0]  
        flow_type = parts[1]       
        
     
        if flow_type not in ['office', 'origin']:
            continue
            
    except IndexError:
        continue

   
    # Proccess only the valid files
    df = pd.read_parquet(file_path)
    
    #  Unpivot Years
    id_vars = [c for c in df.columns if not c.isdigit()]
    
    df_melted = df.melt(
        id_vars=id_vars, 
        var_name="year", 
        value_name="count"
    )
    
    # Harmonize Columns
    if flow_type == "office":
        df_melted.rename(columns={"Office": "office", "Type": "type"}, inplace=True)
        df_melted["origin"] = "Total"
        
    elif flow_type == "origin":
        df_melted.rename(columns={"Origin": "origin", "Type": "type"}, inplace=True)
        df_melted["office"] = "Total"

    # Add Indicator
    df_melted["indicator"] = indicator_code
    
    # Final Selection
    final_cols = ["year", "office", "origin", "type", "indicator", "count"]
    

    if not all(col in df_melted.columns for col in final_cols):
        continue
        
    df_ready = df_melted[final_cols].copy()
    
    #  Data Types
    df_ready["year"] = df_ready["year"].astype(int)
    df_ready["count"] = pd.to_numeric(df_ready["count"], errors="coerce")
    
    all_dfs.append(df_ready)

In [ ]:
if all_dfs:
    fct_ip_flows = pd.concat(all_dfs, ignore_index=True)
    fct_ip_flows = fct_ip_flows.dropna(subset=["count"])
    
    # Preview
    print(f"Processed {len(all_dfs)} valid files.")
    print(f"Total Rows: {len(fct_ip_flows)}")
    print(fct_ip_flows.head(-1))
    
    # Save
    out_file = intermediate_path / "int_ip_flows_by_residency.parquet"
    fct_ip_flows.to_parquet(out_file, index=False)
    print(f"\nSaved final table to: {out_file}")
else:
    print("No valid files (Code_Flow format) were found.")